# Parameter values, methods and settings

<div class="alert alert-info">

**Summary**

This lesson binds the consumption stage of the portfolio–consumption model to its parameter values, numerical methods and numerical settings, and reads after each binding what changes on the card of the decision builder and on the kernel of its aggregation. The object inspected is the stage at its four versions. The lesson evaluates one kernel at one point and solves nothing.

</div>

<div class="alert alert-success">

**Relationship with other documents**

**Builds on** — [The stage as an object](01-the-stage.ipynb) and [Value functions and the backward builders](02-value-functions.ipynb): the stage and its cards.

**Formal account** — [Methodization](../../docs/Bellman-Sym/04-methodization.md): where a method may stand in a stage, the options it reads and the settings that give them values.

**Used by** — [Periods and trellises](05-periods-and-trellises.ipynb): the three bindings of every stage of a trellis.

**See also** — [The portfolio–consumption application](../../applications/cons-port/README.md): the files read here.

</div>

<div class="alert">

**Key terminology**

| Term | Meaning |
|---|---|
| [[symbolic_stage\|symbolic]], [[specified_stage\|specified]] | Said of a stage with no binding, and with all three. |
| place | Where in a stage a method may stand, such as a builder, a line of a block, an operation or a declared function. |
| addition | A `.bl` file beside the stage file, `methods.bl`, written in the language of the stage file, whose typing entries name a method at places of the stage and whose lines are read with the stage file's. |
| typing entry | A line of an addition, such as `policy.method : !egm`, that names at one place one part of a builder's signature: the method word, the grid or the handler. |
| [[handler]] | The function of an application's tool that carries a method out. |

</div>

The [[stage]] is the consumption stage of the portfolio–consumption application, written with the maximization alone: the household holds resources $\mathrm m$ at its decision, chooses consumption $\mathrm c$ in $(0, \mathrm m]$ and carries savings $\mathrm a = \mathrm m - \mathrm c$ forward, with utility $u(\mathrm c) = \mathrm c^{1-\rho}/(1-\rho)$ and relative risk aversion $\rho$.

The stage file declares $\rho$ but gives it no value, and it names no grid and no method of solution. Three files supply these, and three verbs of the stage bind them, in this order: `calibrate` binds the values of the [[parameters]], `methodize` reads the addition, the `.bl` file beside the stage file whose typing entries name a numerical method at places of the stage, and `configure` gives values to the [[settings]] the addition declares and the methods read.

The lesson reads five files and writes none:

- `applications/cons-port/declaration/stages/cons/cons.bl`
- `applications/cons-port/declaration/calibration/portfolio.yml`
- `applications/cons-port/declaration/stages/cons/methods.bl`
- `applications/cons-port/declaration/stages/cons/methods.vfi.bl`
- `applications/cons-port/declaration/settings/cons.yml`

The first code cell imports the collection's set-up module, checkout, which the first lesson describes, the three names of the package the lesson calls, and the package's reader of a calibration file, load_calibration of the module bellman.stage.sources.

In [1]:
import checkout
from bellman.stage import elaborate_stage, load_stage, method_types
from bellman.stage.sources import load_calibration

## The symbolic stage

The next cell constructs the stage from its file, as the first lesson does, and names it `cons`.

In [2]:
cons = elaborate_stage(load_stage("applications/cons-port/declaration/stages/cons/cons.bl"))

`par` answers a table of the parameters the file declares, each with the set its declaration names, its value and, under `describes`, the description a declaration may write after `::`, none here. The file's line `ρ : [1, inf)` declares $\rho$ in $[1, \infty)$, and no line gives it a value.

In [3]:
cons.par

name  set       value    describes
ρ     (1, inf)  unbound

A declaration writes its description as a quoted entry in the bracketed list a line may carry after `::`, the list in which the line defining utility writes its identifier, `[id=util]`. The next cell constructs the stage again from a copy of the file's text in which the declaration of $\rho$ reads `ρ : (1, inf)  :: ["relative risk aversion"]`, and names it `cons_described`; the file itself is unchanged. The cell gives `elaborate_stage` the copy in the form `load_stage` returns, a mapping that holds under `stage` the text and a name for its origin, and holds no calibration, no settings and, under `methods`, an empty list, since a record holds its methods as the additions read into it.

In [4]:
cons_described = elaborate_stage({
    "stage": {
        "text": open("applications/cons-port/declaration/stages/cons/cons.bl").read().replace(
            "ρ : (1, inf)", 'ρ : (1, inf)  :: ["relative risk aversion"]'),
        "origin": "cons-described.bl",
    },
    "calibration": {},
    "settings": {},
    "methods": [],
})

Its table of parameters prints the description under `describes`, and `cons_described.par["ρ"].describes` answers the same text for $\rho$ alone.

In [5]:
cons_described.par

name  set       value    describes
ρ     (1, inf)  unbound  relative risk aversion

`elaborate_stage` binds the parameter values a record holds under `calibration`, as the verb `calibrate` of the section on parameter values below binds them, and answers the symbolic stage, as above, for a record that holds none: the next cell gives it the same text with `applications/cons-port/declaration/calibration/portfolio.yml` read into `calibration`, and its table of parameters holds the value of $\rho$ beside the description.

In [6]:
elaborate_stage({
    "stage": {
        "text": open("applications/cons-port/declaration/stages/cons/cons.bl").read().replace(
            "ρ : (1, inf)", 'ρ : (1, inf)  :: ["relative risk aversion"]'),
        "origin": "cons-described.bl",
    },
    "calibration": load_calibration("applications/cons-port/declaration/calibration/portfolio.yml"),
    "settings": {},
    "methods": [],
}).par

name  set       value  describes
ρ     (1, inf)  6.0    relative risk aversion

A parameter is named in the definitions that use it. The next cell asks for the card of utility, the printed statement of that definition, by its key. Under `Conditions` the card records one condition the construction does not verify: $1 - \rho$ is not zero, since the formula divides by it.

In [7]:
cons.definitions.denote("cons:function:𝑢")

STAGE cons / FUNCTION 𝑢
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Defines
  𝑢  : ℝ₊₊ → (−∞, 0),  c ↦ c^(1 − ρ)/(1 − ρ)
──────────────────────────────────────────────────────────────────────────────
Source  cons.bl
  5  𝑢(c : ℝ₊₊) := c^(1-ρ)/(1-ρ)

Definitions referenced
  ρ     cons:parameter:ρ
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

`bindings` shows the calibration and the settings of the version, each absent here and printed beside the verb that supplies it; the methods an addition types are listed by `methods`, in the section on methods below.

In [8]:
cons.bindings

calibration   none   calibrate(stage, source)
settings      none   configure(stage, source)

## Parameter values

A calibration file gives parameter values as one mapping. This one serves the four stages of the model, so it holds more names than the consumption stage declares.

In [9]:
print(open("applications/cons-port/declaration/calibration/portfolio.yml").read())

# The parameter values of the portfolio-consumption model, those the fixed reference was computed with
# (reference/legacy/syntax/calibration.yaml). One mapping serves the four stages; each stage binds the parameters it declares.
calibration:
  β: 0.96
  ρ: 6.0
  R: 1.02
  μ_Ψ: 0.028927
  σ_Ψ: 0.143486
  μ_θ: -0.01125
  σ_θ: 0.15
  ρ_ζ: 0.0



[[calibration|Calibration]] binds the values. `calibrate` checks the value of each declared parameter against its set and, at those values, decides each condition that involves only parameters; it refuses a value outside its set or one that makes a condition false. Of the file's names it binds `ρ` alone. It binds the stage it is called on and returns it; the next cell names the result `cons_calibrated`, a second name of the same stage.

In [10]:
cons_calibrated = cons.calibrate("applications/cons-port/declaration/calibration/portfolio.yml")

Its table of parameters carries the value. One parameter is read by the name the file declares: `cons_calibrated.par["ρ"]` answers the record of $\rho$ alone, and its `value` is the number bound, 6.0.

In [11]:
cons_calibrated.par


name  set       value  describes
ρ     (1, inf)  6.0

Written as an attribute, the name answers the value alone: the next cell shows the record `cons_calibrated.par["ρ"]` beside `cons_calibrated.par.ρ`, the number 6.0. On the symbolic `cons`, which binds no value, `cons.par.ρ` is refused and the record is read by the key.

In [12]:
cons_calibrated.par["ρ"], cons_calibrated.par.ρ


(ρ = 6.0 : (1, inf), 6.0)

The next cell prints the card of utility on the calibrated version. The formula still names $\rho$, whose value the version holds in its bindings, beside the definition. The condition on $1 - \rho$ no longer stands under `Conditions`: calibration decided it at $\rho = 6.0$, where it holds.

In [13]:
cons_calibrated.definitions.denote("cons:function:𝑢")

STAGE cons / FUNCTION 𝑢
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Defines
  𝑢  : ℝ₊₊ → (−∞, 0),  c ↦ c^(1 − ρ)/(1 − ρ)
──────────────────────────────────────────────────────────────────────────────
Source  cons.bl
  5  𝑢(c : ℝ₊₊) := c^(1-ρ)/(1-ρ)

Definitions referenced
  ρ     cons:parameter:ρ
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The verb bound the stage it was called on and returned it, so the two names show one version, as their short forms show.

In [14]:
cons, cons_calibrated

(ElaboratedStage('cons', calibrated), ElaboratedStage('cons', calibrated))

## The kernel at the values

A **kernel** is the mathematical data an operation uses: a law of motion for a pullback or a push-forward, a value aggregator for an aggregation, a probability distribution for an expectation. The decision builder's aggregation applies $G(v, \mathrm m, \mathrm c) = u(\mathrm c) + v$, which gives the value $V$ at the decision when $v$ is the pulled-back continuation value. On the calibrated version the kernel is the line of the symbolic version, with $\rho$ still a name inside $u$.

In [15]:
cons_calibrated.dcsn.builder.bwd["Aggregation"].kernel()

𝐺 of cntn_to_dcsn: 𝐺(V≻, m, c) = 𝑢(c) + V≻
output V; unused m

`compile_kernel()` turns the kernel into a Python function whose arguments are the kernel's own, in the printed order, followed by each parameter the kernel reads, here $\rho$. The answer of `kernel()` is a record, displayed as the rule above. Under `parameter_values` it holds the calibration's values, here `(6.0,)`, which a caller passes to the function at each call; under `python` it holds the function's source text, which the next cell prints. A symbolic version compiles too, and its record holds `None` under `parameter_values`.

In [16]:
print(cons_calibrated.dcsn.builder.bwd["Aggregation"].kernel().python)

# bellman kernel of cons, cntn_to_dcsn:Aggregation; cons.bl sha256 34cf564877c0d7a4ecd5e15dcd55ed776ffe7f649a1cb674d378753b33cdea78; bellman 0.1.7
# 𝐺 of cntn_to_dcsn: 𝐺(V≻, m, c) = 𝑢(c) + V≻
# output V; unused m
# parameters: ρ ∈ (1, inf)
# names: V_cntn is V≻; u is 𝑢
import numpy as np

def G_cntn_to_dcsn(V_cntn, m, c, ρ):
    def u(c):
        return c**(1 - ρ)/(1 - ρ)
    V = u(c) + V_cntn
    return V



The next cell evaluates the compiled kernel at continuation value $V_\succ = 4$, resources $\mathrm m = 2$ and consumption $\mathrm c = 0.5$.

In [17]:
print(cons_calibrated.dcsn.builder.bwd["Aggregation"].compile_kernel()(4.0, 2.0, 0.5, *cons_calibrated.dcsn.builder.bwd["Aggregation"].kernel().parameter_values))

-2.4000000000000004


The value is $u(0.5) + 4 = 0.5^{-5}/(-5) + 4 = -2.4$, printed as $-2.4000000000000004$ because $-6.4$ has no exact binary representation and the sum keeps its rounding. The kernel has one output: the stage file writes no marginal value, and the addition of the next section states the marginal value $u'(\mathrm c)$ by a derivative line the author writes, which it appends to the block `cntn_to_dcsn`.

## Methods

A methods file is an **addition**: a `.bl` file beside the stage file, `methods.bl`, written in the language of the stage file, which [[methodization]] reads with the stage file as one text. It opens with `@methods: cons`, the name of the stage it adds to. Its **typing entries** name, at a place of the stage, one part of a builder's signature: `method` names the numerical method by its word, `grid` the grid on which the method computes or stores its function, and `handler` the [[handler]], the function of the application's numerical tool that carries the method out. The next cell prints the file. After the header come two definitions, the marginal utility and its inverse in closed form, so that nothing is differentiated, and six declarations of settings, each with the set its value must lie in and a description. The block `policy` names the stage's policy block by its id, and its header carries three typing entries: `method : !egm`, the endogenous-grid method on the policy builder; `handler : "cons-port:egm"`, the function `egm` of the tool; and `grid : !Cartesian(a=(a_min, a_max, n_a, !expmult))`, the grid of savings on which the method computes, one factor for the coordinate `a` whose bounds and count are the names of settings and whose spacing is the word `!expmult`, multi-exponential and dense near zero. Its three lines are the method's: the candidate consumption at each saving from the inverse Euler equation with its clause, the resources at which that saving is chosen, and the policy read on those resources. The block `cntn_to_dcsn` names the decision builder's block; its header types the Evaluation that closes that builder with the representation `!Linear`, linear interpolation between stored values, on the grid of resources, and its one line is the envelope condition, the derivative of the decision value.

In [18]:
print(open("applications/cons-port/declaration/stages/cons/methods.bl").read())

@methods: cons
# The methods of the consumption stage at the occurrences of the two recipes,
# spec.cons-port.yml and spec.port-cons.yml, whose settings file
# settings/cons_wide.yml values multi-exponential grids: the endogenous grid
# method on the policy builder, realized by the function egm of
# code/cons_port_tool.py on the savings grid, with its three lines, the inverse
# Euler equation, the endogenous grid of resources and the identification of
# the policy; the envelope line of the decision value; and one linear
# representation of that value and of its stated derivative on the grid of
# resources. The marginal utility and its inverse are defined in closed form,
# so that nothing is differentiated. The three lines of the policy block stand
# in place of the maximization at the operative level, and stage.declared keeps
# the maximization. Each grid is its own typing entry at its place, the word
# !expmult its spacing; methods.reference.bl writes the same methods on evenly
# spac

`places()` lists every address at which a typing entry may stand, with its kind: the three builders; the named line `policy.optimality`, the maximization, of kind `statement`; the operator `argmax` in that line, by its address in the line, by its address in the block and by its stage-wide name; the Evaluations that close the decision builder and the policy builder, `cntn_to_dcsn.evaluate` and `policy.evaluate`, of kind `operator evaluate`, where a representation is bound, each refined by the object it stores, `cntn_to_dcsn.evaluate[V]` for the value function and `policy.evaluate[c]` for the policy function; and the declared function $u$ and the correspondence $\Gamma$, of kind `definition`. The subject of a typing entry names one of them. The lines an addition appends add places of their own, the lines and the operators they hold, which the specified version lists at the end of the lesson.

In [19]:
cons.places()

address                   kind
dcsn_to_cntn              builder
cntn_to_dcsn              builder
cntn_to_dcsn.evaluate     operator evaluate
cntn_to_dcsn.evaluate[V]  operator evaluate
policy                    builder
policy.optimality         statement
policy.optimality.argmax  operator argmax
policy.argmax             operator argmax
policy.evaluate           operator evaluate
policy.evaluate[c]        operator evaluate
argmax                    operator argmax
𝑢                         definition
Γ                         definition

`methodize` reads the addition into the calibrated version: the stage is elaborated anew from the union of its file with the addition, the typing entries bound at their places and the lines appended to their blocks, and the calibration stays bound. It binds the stage it is called on and returns it; the next cell names the result `cons_methodized`.

In [20]:
cons_methodized = cons_calibrated.methodize("applications/cons-port/declaration/stages/cons/methods.bl")

`methods` lists each place a typing entry types, with what is bound there as the package prints it: the method word, the grid as the entry writes it and the handler in quotation marks. `!egm` stands on the builder `policy` with its grid of savings and its handler; the block's three lines stand in that builder in place of the maximization, which the declared builders, `cons_methodized.declared`, keep. `!Linear` stands at `cntn_to_dcsn.evaluate`, the Evaluation that closes the decision builder, with its grid of resources; the value and its stated derivative, the envelope line the addition appends, are stored there together.

In [21]:
cons_methodized.methods

cntn_to_dcsn.evaluate   !Linear   !Cartesian(m=(m_min, m_max, n_m, !expmult))
policy                  !egm   !Cartesian(a=(a_min, a_max, n_a, !expmult))   "cons-port:egm"

The Evaluation that closes the policy builder reads the consumption rule $c$ from the block's lines: at each level of savings $\mathrm a$, the consumption $c_\succ(\mathrm a)$ the inverse Euler equation allows and the resources $m_{\succ}(\mathrm a) = \mathrm a + c_\succ(\mathrm a)$ that lead to it. The first line of the card's `Rule` states $c$ on the resources so reached, $m_{\succ}(\mathrm{Xa})$, the endogenous grid; the second, the declared set of feasible rules, into which the evaluation returns $c$ unchanged. The method stands on the builder and not on this evaluation, so the card has no `Method` heading; the builder's card, `cons_methodized.dcsn.builder.policy.denote()`, prints it.

In [22]:
cons_methodized.dcsn.builder.policy["Evaluation"].denote()

STAGE cons / POLICY BUILDER policy / Evaluation
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑐  : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)
       ───────────────────────────
  out  𝑐  : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Rule
  𝑐(𝑚≻(a)) = 𝑐≻(a)                                                   on 𝑚≻(Xa)
  𝑐 : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)
──────────────────────────────────────────────────────────────────────────────
Definitions referenced
  Xm    cons:set:Xm
  Xa    cons:set:Xa
  Γ     cons:correspondence:Γ
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The card of the decision builder gains, under `Operations`, the stated derivative at its Evaluation, `D_{m}𝑉(m) = 𝑑𝑢(𝑐(m))`, the envelope line the addition appends; below `Conditions`, a `Method` heading with the representation `!Linear`, its grid as the typing entry writes it, by the names of its settings, which have no values yet, and the place it stands at; and under `Source` the lines of both files, the value line of `cons.bl` and the envelope line of `methods.bl`, whose marginal utility `Definitions referenced` locates in the addition.

In [23]:
cons_methodized.dcsn.builder.bwd.denote()

STAGE cons / BACKWARD BUILDER cntn → dcsn (cntn_to_dcsn)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ───────────────────────────────────────────────────
  out  𝑉   : Xm → ℝ

At
  𝑐  : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻(·, 𝑐)
     ─────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻(·, 𝑐))(m) = 𝑉≻(𝑔≻(m, 𝑐(m)))

  2  Aggregation
     in: (1)                            out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉(m) = 𝑢(𝑐(m)) + 𝑉≻(𝑔≻(m, 𝑐(m)))

  3  Evaluation
     in: 𝑉                              out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉 : Xm → ℝ
           D_{m}𝑉(m) = 𝑑𝑢(𝑐(m))

Conditions
  Aggregation  𝑉(m) ∈ ℝ

Method
  !Linear   !Cartesian(m=(m_min, m_max, n_m, !expmult))  cntn_to_dcsn.evaluate
─────────────────────────────────────────────────────────────

`method_types` lists the method words the package knows. The entry of a word prints as a card that gives, under `Admits`, the kinds of place at which the word may stand, under `Keywords`, the keyword arguments it reads, and under `Grid`, the coordinates of the grid an entry writes beside it: for `!Linear`, four kinds of place, no keyword, and a grid on the states of the perch at which the represented function stands. Of the four kinds, `node` is a line marked `!node`, `statement` a value line, `definition` a declared function, and `operator evaluate` an Evaluation: an authored `evaluate(…)`, or the Evaluation that closes a builder, which an entry names `<block-id>.evaluate`, refined by the object stored where a builder stores several, `cntn_to_dcsn.evaluate[V]`.

In [24]:
method_types["!Linear"]

METHOD !Linear
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Admits
  node
  statement
  definition
  operator evaluate

Keywords
  none

Grid
  on the states of the perch at which the function the place holds stands
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

A method is refused at a place of a kind it does not admit. Gauss–Hermite quadrature computes an expectation; an addition whose entry types the builder `policy` with it is refused. The next cell gives `methodize` such an addition as a mapping of its text and its origin, the form in which a record holds an addition, in place of a file. The refusal names the entry, the place and its kind, the kind of place the word's row of the catalogue admits, an expectation operator, of which this stage has none, and the rule.

In [25]:
cons_calibrated.methodize({"text": "@methods: cons\npolicy.method : !gauss_hermite\n", "origin": "refused.bl"})

ConstructionError: refused.bl:2:1-2:31: the entry `policy.method : !gauss_hermite` types `policy`, a place of the kind builder, with `!gauss_hermite`, whose row of the catalogue admits operator E; write the word at a place of a kind it admits, or a word that admits this place (address 'policy.method'; rule method-admission)
  write: Write the entry at a place of a kind `!gauss_hermite` admits, or give `policy` a word that admits the kind builder

A second addition, `methods.vfi.bl`, is a replacing selection, named by the trellis file of the third lesson of this model in place of `methods.bl`. It appends no line, so it writes no block: each of its entries is a file-level line, routed to its place by the path it begins with. It types the decision builder `cntn_to_dcsn` with `!vfi`, a grid search over candidate consumption levels, with its handler and its grid, `!Cartesian(c=(Γ, n_c))`, one factor for the control whose points are `n_c` candidates on the feasible set $\Gamma$, and the Evaluation of that builder with `!Linear` on the grid of resources.

In [26]:
print(open("applications/cons-port/declaration/stages/cons/methods.vfi.bl").read())

@methods: cons
# The methods of the consumption stage at the occurrences of
# trellis.port-cons-vfi.yml, whose settings file settings/cons_vfi.yml values a
# multi-exponential grid of resources: value function iteration on the decision
# value builder, realized by the function consumption_search of
# code/cons_port_tool.py, which compares n_c candidate levels of consumption on
# the feasible set at each point of the grid of resources; and the linear
# representation of the decision value on that grid. No line is added, so no
# block is written: each selection is a file-level line routed by its path.
# methods.vfi_reference.bl writes the same methods on an evenly spaced grid.

n_m : ℕ              :: ["number of points of the grid of resources"]
m_min : Xm           :: ["lower bound of the grid of resources"]
m_max : Xm           :: ["upper bound of the grid of resources"]
n_c : ℕ              :: ["number of candidate levels of consumption"]

cntn_to_dcsn.method : !vfi
cntn_to_dcsn.hand

`methodize` binds the stage it is called on, and a call is the whole selection of the stage, replacing the additions read before; so the next cell binds the second file on a stage of its own, elaborated and calibrated again from the same files, and names it `cons_methodized_vfi`.

In [27]:
cons_methodized_vfi = elaborate_stage(load_stage("applications/cons-port/declaration/stages/cons/cons.bl")).calibrate(
    "applications/cons-port/declaration/calibration/portfolio.yml").methodize("applications/cons-port/declaration/stages/cons/methods.vfi.bl")

`!vfi` stands on the decision builder `cntn_to_dcsn`, where the addition types it, with the handler `consumption_search` and its grid of candidates: at each level of resources the grid search maximizes over candidate consumption levels the value that builder computes, $u(\mathrm c) + V_\succ(\mathrm m - \mathrm c)$. The policy builder keeps the maximization the stage file writes, since this addition appends no line to it.

In [28]:
cons_methodized_vfi.methods

cntn_to_dcsn            !vfi   !Cartesian(c=(Γ, n_c))   "cons-port:consumption_search"
cntn_to_dcsn.evaluate   !Linear   !Cartesian(m=(m_min, m_max, n_m, !expmult))

## Settings

A settings file gives values to the names the addition declares: here the grid of resources, 100 points from 0.01 to 10.0, and in its last three names the grid of savings of the endogenous-grid method, as its comment says.

In [29]:
print(open("applications/cons-port/declaration/settings/cons.yml").read())

# The consumption stage's numerical settings: the grid of resources on which the value is represented, the
# default the earlier helper took (ffp_utils.py line 431), and the savings grid of the endogenous-grid method
# (ffp_utils.py lines 713 to 715).
settings:
  n_m: 100
  m_min: 0.01
  m_max: 10.0
  n_a: 800
  a_min: 1.0e-6
  a_max: 100.0



`configure` binds the settings to the methodized version, which the next cell names `cons_specified`, the **specified** version. It values each setting the addition declares, checking the value against the set declared for its name, `ℕ` for a count and `Xm`, the set of resources, for a bound of the grid of resources; a declared setting without a value, and a value outside its set, are refused.

In [30]:
cons_specified = cons_methodized.configure("applications/cons-port/declaration/settings/cons.yml")

`settings` lists each setting with the set the addition declares, its value, the place that reads it, the method that reads it there and, under `grid`, the kind of the grid of which the setting is a bound or a count. The first three are read at `policy` by `!egm` as its grid of savings, and the last three at `cntn_to_dcsn.evaluate` by `!Linear` as its grid of resources; both grids are of the kind `!Cartesian`. One setting is read by its name in the same way, `cons_specified.settings["n_m"]`, whose `value` is 100.

In [31]:
cons_specified.settings

name   set  value  read at                by       grid
n_a    ℕ    800    policy                 !egm     !Cartesian
a_min  Xa   1e-06  policy                 !egm     !Cartesian
a_max  Xa   100.0  policy                 !egm     !Cartesian
n_m    ℕ    100    cntn_to_dcsn.evaluate  !Linear  !Cartesian
m_min  Xm   0.01   cntn_to_dcsn.evaluate  !Linear  !Cartesian
m_max  Xm   10.0   cntn_to_dcsn.evaluate  !Linear  !Cartesian

A setting's name written as an attribute answers its value alone, as a parameter's does: the next cell shows the record `cons_specified.settings["n_a"]` beside `cons_specified.settings.n_a`, the number 800.

In [32]:
cons_specified.settings["n_a"], cons_specified.settings.n_a

(n_a = 800 : ℕ, read at policy by !egm for its grid !Cartesian, 800)

The next cell prints the decision builder's card on the specified version. The card is unchanged: the method's grid is written as the typing entry writes it, by the names of its settings, and the values stand in the settings table alone.

In [33]:
cons_specified.dcsn.builder.bwd.denote()

STAGE cons / BACKWARD BUILDER cntn → dcsn (cntn_to_dcsn)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ───────────────────────────────────────────────────
  out  𝑉   : Xm → ℝ

At
  𝑐  : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻(·, 𝑐)
     ─────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻(·, 𝑐))(m) = 𝑉≻(𝑔≻(m, 𝑐(m)))

  2  Aggregation
     in: (1)                            out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉(m) = 𝑢(𝑐(m)) + 𝑉≻(𝑔≻(m, 𝑐(m)))

  3  Evaluation
     in: 𝑉                              out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉 : Xm → ℝ
           D_{m}𝑉(m) = 𝑑𝑢(𝑐(m))

Conditions
  Aggregation  𝑉(m) ∈ ℝ

Method
  !Linear   !Cartesian(m=(m_min, m_max, n_m, !expmult))  cntn_to_dcsn.evaluate
─────────────────────────────────────────────────────────────

The aggregation's kernel on the specified version is the line of the section [The kernel at the values](#The-kernel-at-the-values): a method and its settings say how a function is computed and stored, and they leave the value line, whose kernel this is, as the stage file writes it.

In [34]:
cons_specified.dcsn.builder.bwd["Aggregation"].kernel()

𝐺 of cntn_to_dcsn: 𝐺(V≻, m, c) = 𝑢(c) + V≻
output V; unused m

## The four versions

The four names, from the symbolic version to the specified one, name one stage, which the verbs bound in place; each short form shows the version the stage has reached now.

In [35]:
cons, cons_calibrated, cons_methodized, cons_specified

(ElaboratedStage('cons', specified),
 ElaboratedStage('cons', specified),
 ElaboratedStage('cons', specified),
 ElaboratedStage('cons', specified))

The next cell shows the specified version's bindings: the value of $\rho$ and the six settings with their values. The methods the addition types are not among them; `methods` lists those.

In [36]:
cons_specified.bindings

calibration   ρ = 6.0
settings      n_m     100
              m_min   0.01
              m_max   10.0
              n_a     800
              a_min   1e-06
              a_max   100.0

The places of the specified version add to those of the symbolic one the lines the addition appended, `cntn_to_dcsn.envelope` and the three lines of the policy block, the derivative operators those lines hold, by their addresses and by the stage-wide name `D`, the Evaluation of the decision builder refined by the stated derivative, `cntn_to_dcsn.evaluate[D_{m}V]`, and the two definitions of the addition.

In [37]:
cons_specified.places()

address                        kind
dcsn_to_cntn                   builder
cntn_to_dcsn                   builder
cntn_to_dcsn.envelope          statement
cntn_to_dcsn.evaluate          operator evaluate
cntn_to_dcsn.evaluate[V]       operator evaluate
cntn_to_dcsn.evaluate[D_{m}V]  operator evaluate
policy                         builder
policy.optimality              statement
policy.optimality.argmax       operator argmax
policy.InvEuler                statement
policy.InvEuler.D              operator D
policy.grid                    statement
policy.identification          statement
policy.argmax                  operator argmax
policy.D                       operator D
policy.evaluate                operator evaluate
policy.evaluate[c]             operator evaluate
argmax                         operator argmax
D                              operator D
𝑢                              definition
Γ                              definition
𝑑𝑢                             definition
𝑑𝑢𝑖𝑛

The package constructs and binds the stage; an application's numerical tool, whose functions the handlers name, solves it.